### Library Imports

In [46]:
#You may need to add other libraries here depending on your code

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# to display plots in Jupyter notebook
%matplotlib inline

plt.rcParams['font.size'] = '12'

### Get California Housing Data & Train Test Split

In [47]:
from pathlib import Path
import pandas as pd
import tarfile
import urllib.request

def load_housing_data():
    tarball_path = Path("datasets/housing.tgz")
    if not tarball_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball_path)
    with tarfile.open(tarball_path) as housing_tarball:
            housing_tarball.extractall(path="datasets")
    return pd.read_csv(Path("datasets/housing/housing.csv"))

housing = load_housing_data()

X = housing.drop(["median_house_value", "ocean_proximity"], axis=1)
y = housing["median_house_value"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Clean up data

In [48]:
imputer = SimpleImputer(strategy="mean")

X_train_cleaned = imputer.fit_transform(X_train)
X_test_cleaned = imputer.transform(X_test)

### Run kNN

In [49]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_cleaned)
X_test_scaled = scaler.transform(X_test_cleaned)

params = {
    "n_neighbors": range(1, 20),
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"]
}

knn = KNeighborsRegressor()

grid_search = GridSearchCV(
    estimator=knn,
    param_grid=params,
    cv=10,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)

print(grid_search.best_params_)
print(-grid_search.best_score_)

{'metric': 'manhattan', 'n_neighbors': 12, 'weights': 'distance'}
59528.27261491285


### Best kNN

In [50]:
best_knn = grid_search.best_estimator_

y_train_pred = best_knn.predict(X_train_scaled)
y_test_pred  = best_knn.predict(X_test_scaled)

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse  = np.sqrt(mean_squared_error(y_test, y_test_pred))

train_mae  = mean_absolute_error(y_train, y_train_pred)
test_mae   = mean_absolute_error(y_test, y_test_pred)

train_r2   = r2_score(y_train, y_train_pred)
test_r2    = r2_score(y_test, y_test_pred)

print(f"{'':20} {'Train':>10} {'Test':>10}")
print(f"{'RMSE':20} {train_rmse:>10.4f} {test_rmse:>10.4f}")
print(f"{'MAE':20} {train_mae:>10.4f} {test_mae:>10.4f}")
print(f"{'R2 Score':20} {train_r2:>10.4f} {test_r2:>10.4f}")

                          Train       Test
RMSE                     0.0000 60694.7029
MAE                      0.0000 41765.9813
R2 Score                 1.0000     0.7343
